# Load and inspect MCAP data

This notebook loads an MCAP file, lists recorded topics, and plots a selected topic.

In [3]:
from __future__ import annotations

from dataclasses import dataclass
import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mcap.reader import make_reader


@dataclass
class McapDataset:
    path: Path
    topics: list[str]
    messages: dict[str, list[dict[str, Any]]]


class McapDataLoader:
    """Load and inspect Foxglove MCAP files written by ExperimentLogger."""

    def __init__(self, mcap_path: str | Path):
        self.path = Path(mcap_path)
        if not self.path.exists():
            raise FileNotFoundError(f"MCAP file not found: {self.path}")

    @staticmethod
    def _flatten(value: Any, prefix: str = "") -> dict[str, Any]:
        out: dict[str, Any] = {}

        if isinstance(value, dict):
            for key, child in value.items():
                key_str = str(key)
                child_prefix = f"{prefix}_{key_str}" if prefix else key_str
                out.update(McapDataLoader._flatten(child, child_prefix))
            return out

        if isinstance(value, (list, tuple)):
            if len(value) == 1 and not isinstance(value[0], (dict, list, tuple)):
                if prefix:
                    out[prefix] = value[0]
                return out
            for i, child in enumerate(value):
                child_prefix = f"{prefix}_{i}" if prefix else str(i)
                if isinstance(child, (dict, list, tuple)):
                    out.update(McapDataLoader._flatten(child, child_prefix))
                else:
                    out[child_prefix] = child
            return out

        if prefix:
            out[prefix] = value
        return out

    def load(self) -> McapDataset:
        messages: dict[str, list[dict[str, Any]]] = {}

        with self.path.open("rb") as handle:
            reader = make_reader(handle)
            for _schema, channel, message in reader.iter_messages():
                topic = channel.topic
                payload = message.data if isinstance(message.data, (bytes, bytearray)) else bytes(message.data)
                record = json.loads(payload.decode("utf-8"))

                flat_values = self._flatten(record.get("values", {}))
                record["values"] = flat_values
                record.update(flat_values)
                record["timestamp_s"] = float(record.get("ts", message.log_time / 1e9))
                record["_log_time_ns"] = int(message.log_time)

                messages.setdefault(topic, []).append(record)

        topics = sorted(messages)
        return McapDataset(path=self.path, topics=topics, messages=messages)


def plot_topic(dataset: McapDataset, topic: str, *, time_key: str = "timestamp_s") -> None:
    records = dataset.messages.get(topic, [])
    if not records:
        raise ValueError(f"No records found for topic: {topic}")

    frame = pd.DataFrame(records)
    if time_key in frame.columns:
        t = frame[time_key].astype(float)
    else:
        t = np.arange(len(frame), dtype=float)

    excluded = {time_key, "stream", "values", "ts", "_log_time_ns"}
    numeric_cols = [
        c for c in frame.columns if c not in excluded and pd.api.types.is_numeric_dtype(frame[c])
    ]
    if not numeric_cols:
        raise ValueError(f"No numeric fields available to plot for topic: {topic}")

    plt.figure(figsize=(12, 5))
    for col in numeric_cols:
        plt.plot(t, frame[col], label=col)

    plt.title(f"{topic} from {dataset.path.name}")
    plt.xlabel(time_key if time_key in frame.columns else "sample index")
    plt.ylabel("value")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()


In [5]:
# Point this at a run produced by ExperimentLogger, e.g.
# data/rim_runs/run_YYYYMMDD_HHMMSS/samples.mcap
data_dir = Path("/home/athena/csirois/data/franka/rim")
run_name = "run_20260424_121938"
mcap_path = data_dir / run_name / "samples.mcap"

loader = McapDataLoader(mcap_path)
dataset = loader.load()

print(f"Loaded: {dataset.path}")
print("Recorded topics:")
for topic in dataset.topics:
    print(f" - {topic} ({len(dataset.messages[topic])} messages)")

# Select a topic to plot.
selected_topic = dataset.topics[0] if dataset.topics else None
print(f"Selected topic: {selected_topic}")

RecordLengthLimitExceeded: unknown (opcode 255) record has length 18446744073709551615 that exceeds limit 4294967296

In [ ]:
if selected_topic is not None:
    plot_topic(dataset, selected_topic)
else:
    print("No topics found in the MCAP file.")